In [1]:
import numpy as np
import pandas as pd
import sklearn.model_selection
import argparse, os, glob, re, warnings, shutil, pysam, vcf, subprocess
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
import statsmodels.api as sm
from Bio import Seq, SeqIO
import scipy.stats as st

df_isolate_metadata = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/isolate_metadata.csv")

personal_ref_dir = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_aln_personal_assembly"
H37Rv_ref_dir = "/n/data1/hms/dbmi/farhat/Sanjana/TRUST_lowAF"

TRUST_data_dir = "~/TRUST_data_processing/processed_data"
df_trust_patients = pd.read_csv(f"{TRUST_data_dir}/combined_patient_WGS_data.csv")

df_assemblies = pd.read_csv("/home/sak0914/MtbLongitudinalDiversity/lowAF_variant_calling/data/TRUST_SR_samples_personal_genomes.csv")

for i, row in df_trust_patients.iterrows():
    if not row['Original_ID'].startswith('S'):
        
        length_numerical = len(row['Original_ID'].split('-')[0])
            
        newID = 'S' + '0' * (4 - length_numerical) + row['Original_ID']
        df_trust_patients.loc[i, 'Original_ID'] = newID
        
    if not pd.isnull(row['Lineage']):
        if type(row['Lineage']) == float:
            df_trust_patients.loc[i, 'Lineage'] = str(int(row['Lineage']))
        else:
            df_trust_patients.loc[i, 'Lineage'] = str(row['Lineage'])
     
print(df_trust_patients.Lineage.value_counts())
            
F2_max = 0.03

mixed_lineage_samples = df_trust_patients.query("F2 > @F2_max").SampleID.unique()
unmixed_lineage_samples = df_trust_patients.query("F2 <= @F2_max").SampleID.unique()

h37Rv_path = "/n/data1/hms/dbmi/farhat/Sanjana/H37Rv"
h37Rv_seq = SeqIO.read(os.path.join(h37Rv_path, "GCF_000195955.2_ASM19595v2_genomic.gbff"), "genbank")
h37Rv_genes = pd.read_csv(os.path.join(h37Rv_path, "mycobrowser_h37rv_genes_v4.csv"))
h37Rv_regions = pd.read_csv(os.path.join(h37Rv_path, "mycobrowser_h37rv_v4.csv"))

h37Rv_coords = pd.read_csv(os.path.join(h37Rv_path, "h37Rv_coords_to_gene.csv"))
h37Rv_coords_dict = dict(zip(h37Rv_coords["pos"].values, h37Rv_coords["region"].values))

# remove rRNAs, which are highly conserved. rrs, rrl, and rrf
rRNA_pos = []

for i, row in h37Rv_regions.query("Functional_Category=='stable RNAs' & Feature=='rRNA'").iterrows():
    print(row['Name'])
    rRNA_pos += list(np.arange(row['Start'], row['Stop'] + 1))
    
    
# exclude any indel within 100 bp of an insertion seq / phage and also within those regions
insertion_seqs_phages_pos = []

for i, row in h37Rv_regions.query("Functional_Category=='insertion seqs and phages'").iterrows():
    
    insertion_seqs_phages_pos += list(np.arange(row['Start'] - 100, row['Stop'] + 100 + 1))
    
insertion_seqs_phages_pos = np.unique(insertion_seqs_phages_pos)
len(insertion_seqs_phages_pos)

def get_id_from_gtf_attribute(attr_string):
    """
    Extract the value of the ID field from a GTF/GFF attributes string.

    Example:
    attr_string = "ID=gene-Rv3324c;Dbxref=GeneID:887981;Name=moaC3;..."
    returns "gene-Rv3324c"
    """
    # Split by semicolon
    fields = attr_string.split(";")
    
    # Iterate through key=value pairs
    for f in fields:
        f = f.strip()
        if f.startswith("ID="):
            return f.split("=", 1)[1]  # split only on first '='
    
    # If no ID field found, return None
    return None


genomic_data_dir = "/n/data1/hms/dbmi/farhat/rollingDB/genomic_data"

Lineage
2      384
4      332
3       48
2,4      6
1        1
2,3      1
Name: count, dtype: int64
rrs
rrl
rrf


In [2]:
def extract_lineages(df, name_col, out_dir=None):

    df_add = []

    if out_dir is None:
        out_dir = "/n/data1/hms/dbmi/farhat/rollingDB/genomic_data"
    
    for sample in df[name_col].unique():

        try:
            flc_fName = f"{out_dir}/{sample}/lineage/fast_lineage_caller_output.txt"
            
            if os.path.isfile(flc_fName):

                F2 = float(pd.read_csv(f"{out_dir}/{sample}/lineage/F2_Coll2014.txt", sep='\t', header=None)[0].values[0])

                df_flc = pd.read_csv(f"{out_dir}/{sample}/lineage/fast_lineage_caller_output.txt", sep='\t')

                df_flc['ROLLINGDB_ID'] = os.path.basename(os.path.dirname(os.path.dirname(flc_fName)))

                for col in df_flc.columns:
                    if col not in ['Isolate', 'ROLLINGDB_ID']:
                        df_flc.rename(columns={col: col.capitalize()}, inplace=True)

                if sum(pd.isnull(df_flc['Coll2014'])) == 0:
                    df_flc['Coll2014'] = df_flc['Coll2014'].str.replace('lineage', '')
                    
                    split_lineages = df_flc['Coll2014'].values[0].split(',')                

                    unique_lineages = []

                    for lineage in split_lineages:
                        if lineage[0].isnumeric():
                            unique_lineages.append(lineage[0])
                        else:
                            unique_lineages.append(lineage)

                    df_flc['Lineage'] = ','.join(np.sort(np.unique(unique_lineages)))
                
                else:
                    df_flc['Lineage'] = ['canettii' if 'canetti' in df_flc['Lipworth2019'].values[0] else np.nan][0]
                
                df_flc['F2'] = F2

                df_add.append(df_flc)
        except:
            print(f"{sample} failed")
    
    df_add = pd.concat(df_add).reset_index(drop=True)
    del df_add['Isolate']
    
    return df_add

In [3]:
# take only public samples
df_samples = pd.DataFrame({'ROLLINGDB_ID': os.listdir(genomic_data_dir)}).query("ROLLINGDB_ID.str.startswith('SAM')")
len(df_samples)

54446

In [ ]:
# df_lineages = pd.read_csv("rollingDB_lineages.csv")
# df_single_lineages = df_lineages.query("F2 <= 0.03")

# exclude_samples = df_single_lineages.dropna(subset='Coll2014').query("Coll2014.str.contains(',')").ROLLINGDB_ID.values

# df_single_lineages = df_single_lineages.query("ROLLINGDB_ID not in @exclude_samples")
# print(len(df_single_lineages))

# df_single_lineages.Lineage.value_counts()

In [ ]:
# df_single_lineages = df_single_lineages.merge(df_isolate_metadata[['ROLLINGDB_ID', 'Num_Runs']], how='left', on='ROLLINGDB_ID')
# len(df_single_lineages)

51305

In [23]:
len(df_single_lineages.query("~(Num_Runs > 1)"))

47783

In [25]:
len(df_single_lineages.query("Num_Runs == 1"))

43425

In [34]:
df_delly_run = df_single_lineages.query("~(Num_Runs > 1)").reset_index(drop=True)
print(len(df_delly_run))

df_delly_run.Lineage.value_counts()

47783


Lineage
4           24896
2           14211
3            5051
1            3294
BOV_AFRI      227
5              58
7              33
canettii       11
6               1
BOV             1
Name: count, dtype: int64

In [35]:
for i, row in df_delly_run.iterrows():
    
    CRAM_files = glob.glob(f"{genomic_data_dir}/{row['ROLLINGDB_ID']}/bam/*.cram")
    assert len(CRAM_files) == 1
    CRAM_files = CRAM_files[0]
    
    df_delly_run.loc[i, 'fName'] = CRAM_files

In [80]:
df_delly_run.query("Lineage not in ['2', '4']")[['ROLLINGDB_ID', 'fName']].to_csv("~/Mtb_Megapipe/group_1_delly.csv", index=False)

In [91]:
df_delly_run.query("Lineage in ['2', '4']")[['ROLLINGDB_ID', 'fName']].to_csv("~/Mtb_Megapipe/group_2_delly.csv", index=False)

# Lineage Distribution of Samples with the Rv2082 Deletion

In [60]:
def read_in_delly_SVs(fName, START, END):
    
    df_SV = []

    vcf_reader = vcf.Reader(filename=fName)

    for record in vcf_reader:
        
        pos = record.POS
        
        if pos >= START and pos <= END:
            
            ref = record.REF
            alt = ''.join([str(a) for a in record.ALT]).strip('<').strip('>')      # ALT is a list of objects
            qual = record.QUAL
            flt = record.FILTER if record.FILTER else "PASS"
            
            info = {k: record.INFO[k] for k in record.INFO}

            if 'IMPRECISE' in info.keys():
                imprecise_bool = True
            else:
                imprecise_bool = False
        
            df_SV.append(pd.DataFrame({'POS': pos, 'REF': ref, 'ALT': alt, 'QUAL': qual, 'FILTER': flt, 'IMPRECISE': imprecise_bool, 
                                       'SVTYPE': info['SVTYPE'], 'END': info['END'], 'MQ': info['MAPQ'], 'PE': info['PE'], 'DV': record.samples[0]['DV'],
                                       'CIPOS_LB': info['CIPOS'][0], 'CIPOS_UB': info['CIPOS'][1], 'CIEND_LB': info['CIEND'][0], 'CIEND_UB': info['CIEND'][1]
                                      }, index=[0])
                        )
    
    if len(df_SV) > 0:
        df_SV = pd.concat(df_SV)
        df_SV['ROLLINGDB_ID'] = os.path.basename(fName).split('.vcf')[0]

        return df_SV.set_index('ROLLINGDB_ID').reset_index()
    else:
        return pd.DataFrame()

In [7]:
df_lineages = pd.read_csv("rollingDB_lineages.csv")
df_single_lineages = df_lineages.query("F2 <= 0.03")

exclude_samples = df_single_lineages.dropna(subset='Coll2014').query("Coll2014.str.contains(',')").ROLLINGDB_ID.values

df_single_lineages = df_single_lineages.query("ROLLINGDB_ID not in @exclude_samples")
print(len(df_single_lineages))

df_single_lineages.Lineage.value_counts()

49348


Lineage
4           25668
2           14748
3            5136
1            3455
BOV_AFRI      234
5              60
7              34
canettii       11
6               1
BOV             1
Name: count, dtype: int64

In [3]:
# 6,802 before
delly_fNames = glob.glob(f"{genomic_data_dir}/*/delly/*.vcf")
len(delly_fNames)

12151

In [4]:
samples_with_delly_results = np.unique([os.path.basename(fName).split('.vcf')[0] for fName in delly_fNames])
len(samples_with_delly_results)

12151

In [45]:
fName

'/n/data1/hms/dbmi/farhat/rollingDB/genomic_data/17-3391-0259/delly/17-3391-0259.vcf'

In [68]:
START = 2338065
END = 2340874

df_SV_Rv2082 = []

for i, fName in enumerate(delly_fNames):
    
    # sample = os.path.basename(fName).split('.DUP.MQ40.vcf')[0]
    
    df_SV_Rv2082.append(read_in_delly_SVs(fName, START, END))
    
    if i % 1000 == 0:
        print(i)
    
df_SV_Rv2082 = pd.concat(df_SV_Rv2082).query("FILTER == 'PASS'").merge(df_single_lineages)
df_SV_Rv2082['SVLEN'] = df_SV_Rv2082['END'] - df_SV_Rv2082['POS']

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000


In [10]:
samples_no_SVs = list(set(samples_with_delly_results) - set(df_SV_Rv2082.ROLLINGDB_ID))
len(samples_no_SVs)

6910

In [11]:
df_single_lineages.query("ROLLINGDB_ID in @samples_no_SVs & Lineage=='3'")

,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,ROLLINGDB_ID,Lineage,F2
487,3,3.1.1,"east_african_indian,ghana",NaN,NaN,SAMEA1018885,3,0.009621
505,3.1.1,3.1.1.i1,east_african_indian,NaN,NaN,SAMEA1018907,3,0.023293
530,3,3.1.1,east_african_indian,NaN,NaN,SAMEA1018936,3,0.010381
556,3,3.1.1,east_african_indian,NaN,NaN,SAMEA1018971,3,0.006042
616,3.1.2,3.1.1.i2,east_african_indian,NaN,NaN,SAMEA1019040,3,0.009401
...,...,...,...,...,...,...,...,...
51305,3,3.1.1,"east_african_indian,indo_oceanic,ghana,mungi",NaN,NaN,SAMN41772239,3,0.012789
51528,3,3.1.1,"east_african_indian,ghana",NaN,NaN,SAMN43780351,3,0.008725
51565,3,3.1.1,"east_african_indian,ghana,dassie",NaN,NaN,SAMN43780394,3,0.008139
51618,3,3.1.1,"east_african_indian,ghana",NaN,NaN,SAMN43780451,3,0.008235


In [ ]:
df_SV_Rv2082.query("Lineage=='2' & SVLEN > 2600 & SVLEN < 3000")

In [18]:
df_SV_Rv2082.query("Lineage=='2' & SVLEN > 2600 & SVLEN < 3000")

,ROLLINGDB_ID,POS,REF,ALT,QUAL,FILTER,IMPRECISE,SVTYPE,END,MQ,PE,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,F2,SVLEN
1506,SAMEA1404043,2338132,G,DUP,180,PASS,True,DUP,2340734,60,3,2.2.1,2.2.1.1.1,"beijing,mungi",lin2.2.1,NaN,2,0.026247,2602
1526,SAMEA1469505,2338129,G,DUP,180,PASS,True,DUP,2340759,60,3,2.2.1,2.2.1.1.1,"beijing,mungi",lin2.2.1,NaN,2,0.011832,2630
7662,SAMN03653272,2338217,A,DEL,10000,PASS,False,DEL,2340923,60,720,2.2.1,2.2.1.1.1,"beijing,canetti,stype,mungi",lin2.2.1,NaN,2,0.006559,2706
8840,SAMN06279759,2339802,C,INV,480,PASS,True,INV,2342778,60,8,2.2.1,2.2.1.1.1,"beijing,canetti","lin2.2.1,asian_african_2",NaN,2,0.007684,2976
8847,SAMN06279759,2339868,T,INV,180,PASS,False,INV,2342832,0,0,2.2.1,2.2.1.1.1,"beijing,canetti","lin2.2.1,asian_african_2",NaN,2,0.007684,2964
8917,SAMN06279763,2340061,T,INV,240,PASS,True,INV,2343055,60,4,2.2.1,2.2.1.1.1,"beijing,canetti",lin2.2.1,NaN,2,0.005425,2994
9821,SAMN08379978,2339769,C,INV,180,PASS,True,INV,2342560,60,3,2.2.1,2.2.1.1.1.i3,beijing,"lin2.2.1,central_asia",NaN,2,0.029136,2791


In [57]:
# df_SV_Rv2082_dup_ins = df_SV_Rv2082.query("SVTYPE in ['DUP', 'INS']")

In [328]:
# sns.histplot(data=df_SV_Rv2082.query("SVTYPE=='DUP'"),
#              x='SVLEN'
#             )

In [307]:
df_SV_Rv2082.query("MQ == 0").SVTYPE.unique()

array(['INV', 'DEL', 'DUP'], dtype=object)

In [36]:
df_SV_Rv2082.SVTYPE.value_counts()

SVTYPE
DUP    4605
INV    1107
DEL     955
Name: count, dtype: int64

In [58]:
len(df_SV_Rv2082_dup_ins.query("SVLEN >= 2000 & SVLEN < 3000")) / len(df_SV_Rv2082_dup_ins)

0.8334419109663409

In [42]:
df_SV_Rv2082.SVTYPE.value_counts()

SVTYPE
DUP    4605
INV    1107
DEL     955
Name: count, dtype: int64

In [66]:
# df_SV_Rv2082_dup_ins.loc[(df_SV_Rv2082_dup_ins['SVTYPE'].isin(['DUP', 'INS', 'DEL'])) & (df_SV_Rv2082_dup_ins['SVLEN'] >= 2000) & (df_SV_Rv2082_dup_ins['SVLEN'] < 3000), 'Rv2082_DUP'] = 1
df_SV_Rv2082_dup_ins.loc[(df_SV_Rv2082_dup_ins['SVTYPE'].isin(['DUP', 'INS', 'DEL'])), 'Rv2082_DUP'] = 1

df_SV_Rv2082_dup_ins['Rv2082_DUP'] = df_SV_Rv2082_dup_ins['Rv2082_DUP'].fillna(0).astype(int)

In [67]:
samples_with_Rv2082_dup = df_SV_Rv2082_dup_ins.query("Rv2082_DUP==1").ROLLINGDB_ID.unique()

samples_without_Rv2082_dup = list(set(samples_with_delly_results) - set(samples_with_Rv2082_dup))

print(f"{len(samples_with_Rv2082_dup)} samples have an Rv2082 duplication")
print(f"{len(samples_without_Rv2082_dup)} samples DO NOT have an Rv2082 duplication")

3997 samples have an Rv2082 duplication
2805 samples DO NOT have an Rv2082 duplication


In [69]:
df_SV_Rv2082_dup_ins.query("ROLLINGDB_ID == @samples_without_Rv2082_dup[0]")

,ROLLINGDB_ID,POS,REF,ALT,QUAL,FILTER,IMPRECISE,SVTYPE,END,MQ,PE,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,F2,SVLEN,Rv2082_DUP


In [71]:
df_SV_Rv2082.query("ROLLINGDB_ID == @samples_without_Rv2082_dup[0]")

,ROLLINGDB_ID,POS,REF,ALT,QUAL,FILTER,IMPRECISE,SVTYPE,END,MQ,PE,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,F2,SVLEN,Rv2082_DUP


In [76]:
df_lineages.query("ROLLINGDB_ID in @samples_with_Rv2082_dup").Lineage.value_counts()

Lineage
3    3011
4     917
5      38
7      31
Name: count, dtype: int64

In [75]:
df_lineages.query("ROLLINGDB_ID in @samples_without_Rv2082_dup").Lineage.value_counts()

Lineage
3           2108
4            617
5             22
canettii      11
7              3
6              1
Name: count, dtype: int64

In [78]:
df_SV_Rv2082.query("ROLLINGDB_ID=='SAMN43780500'")

,ROLLINGDB_ID,POS,REF,ALT,QUAL,FILTER,IMPRECISE,SVTYPE,END,MQ,PE,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,F2,SVLEN,Rv2082_DUP
6651,SAMN43780500,2338269,G,INV,240,PASS,False,INV,2338368,0,0,3,3.1.1,"east_african_indian,ghana",NaN,NaN,3,0.010918,99,0
6652,SAMN43780500,2339530,C,INV,420,PASS,False,INV,2339584,0,0,3,3.1.1,"east_african_indian,ghana",NaN,NaN,3,0.010918,54,0


In [50]:
df_SV_Rv2082_to_analyze = df_SV_Rv2082.sort_values(['ROLLINGDB_ID', 'Rv2082_DUP'], ascending=[True, False]).drop_duplicates('ROLLINGDB_ID', keep='first')

In [51]:
df_SV_Rv2082_to_analyze.groupby('Coll2014')['Rv2082_DUP'].max()

Coll2014
3          1
3.1.1      1
3.1.2      1
3.1.2.1    1
3.1.2.2    1
4.1.1.2    1
4.1.2      1
4.2.2      1
5          1
7          1
Name: Rv2082_DUP, dtype: int64

In [317]:
df_SV_Rv2082_to_analyze

,ROLLINGDB_ID,POS,REF,ALT,QUAL,FILTER,IMPRECISE,SVTYPE,END,MQ,...,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,F2,Lineage,VCF,SVLEN,Rv2082_DUP
0,17-3391-0259,2338129,G,DUP,190,PASS,True,DUP,2340740,54,...,3,3.1.1,"east_african_indian,canetti,mungi",NaN,NaN,0.021228,3,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,2611.0,1
1,1741,2338129,G,DUP,823,PASS,True,DUP,2340759,60,...,4.1.2,4.1.i1.1.2,haarlem,NaN,4,0.021876,4,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,2630.0,1
2,1766,2338129,G,DUP,1960,PASS,True,DUP,2340759,60,...,3,3.1.1,"east_african_indian,canetti",NaN,NaN,0.013323,3,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,2630.0,1
3,1768,2338129,G,DUP,2127,PASS,True,DUP,2340759,60,...,3,3.1.1,east_african_indian,NaN,NaN,0.007388,3,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,2630.0,1
4,1789,2338129,G,DUP,1286,PASS,True,DUP,2340759,60,...,3.1.1,3.1.1.i1,east_african_indian,NaN,NaN,0.013414,3,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,2630.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12844,SAMN43780600,2338129,G,DUP,1680,PASS,True,DUP,2340759,60,...,4.1.1.2,4.1.i1.2.1,"xtype,canetti",NaN,4,0.021002,4,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,2630.0,1
539,SAMN43780691,2338129,G,DUP,1140,PASS,True,DUP,2340759,60,...,3.1.1,3.1.1.i1,"east_african_indian,ghana,canetti,mungi",NaN,NaN,0.008726,3,NaN,NaN,0
12845,TN13542,2338129,G,DUP,300,PASS,True,DUP,2340759,60,...,3,3.1.1,"east_african_indian,ghana,mungi",NaN,NaN,0.018324,3,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,2630.0,1
12846,TN33059,2338129,G,DUP,656,PASS,True,DUP,2340759,60,...,3.1.2,3.1.1.i2,"east_african_indian,ethiopian,ghana,mungi",NaN,NaN,0.013451,3,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,2630.0,1


In [109]:
df_delly = pd.read_csv("/home/sak0914/Mtb_Megapipe/group_1_delly.csv")
len(df_delly)

6791

In [111]:
df_delly = df_delly.merge(df_isolate_metadata, how='left')

In [115]:
df_delly.query("Num_Runs > 1").drop_duplicates(subset='ROLLINGDB_ID').Num_Runs.value_counts()

Num_Runs
2.0    94
4.0    17
3.0     5
Name: count, dtype: int64

In [121]:
df_delly.query("Num_Runs > 2 & ROLLINGDB_ID in @weird_samples").sort_values('Num_Runs').drop_duplicates(subset='ROLLINGDB_ID')[['ROLLINGDB_ID', 'Num_Runs']]

,ROLLINGDB_ID,Num_Runs
4019,SAMN02381042,4.0
4069,SAMN02414941,4.0
4077,SAMN02414945,4.0
4091,SAMN02414962,4.0


In [98]:
df_rerun = pd.DataFrame({'A': ['SAMN02414941', 'SAMN02414945'],
                         'B': ['SRR1162822,SRR1166828', 'SRR1163207,SRR1163493'],
                         'C': [1]*2
                        })

In [99]:
df_rerun.to_csv("~/Mtb_Megapipe/isolates_to_run.tsv", sep='\t', header=None, index=False)

In [165]:
sample = 'SAMN02414962'

for fName in glob.glob(f"/n/data1/hms/dbmi/farhat/rollingDB/genomic_data/{sample}/*/bam/*.cram"):
    print(f"samtools index {fName}")
    print(f"delly call -g $ref_fasta {fName} > {os.path.dirname(fName)}/{os.path.basename(fName).split('.')[0]}.SV.vcf")

samtools index /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1163201/bam/SRR1163201.markDup.cram
delly call -g $ref_fasta /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1163201/bam/SRR1163201.markDup.cram > /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1163201/bam/SRR1163201.SV.vcf
samtools index /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1163208/bam/SRR1163208.markDup.cram
delly call -g $ref_fasta /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1163208/bam/SRR1163208.markDup.cram > /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1163208/bam/SRR1163208.SV.vcf
samtools index /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1165432/bam/SRR1165432.markDup.cram
delly call -g $ref_fasta /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1165432/bam/SRR1165432.markDup.cram > /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMN02414962/SRR1165432/bam/SRR1165432

In [154]:
check_samples = df_isolate_metadata.query("Num_Runs > 2").ROLLINGDB_ID.unique()
len(check_samples)

455

In [157]:
df_metadata = pd.read_csv("~/data_cleaning/data/more_than_2_runs_metadata.csv")

In [156]:
# pd.Series(check_samples).to_csv("~/data_cleaning/data/more_than_2_runs.csv", index=False, header=None)

In [164]:
df_metadata.query("Query=='SAMN02381042'").query("Query in @check_samples")[['LibraryName', 'LibraryStrategy', 'SampleName', 'Sample', 'LibrarySource', 'LibraryLayout']]

,LibraryName,LibraryStrategy,SampleName,Sample,LibrarySource,LibraryLayout


In [122]:
df_isolate_metadata.query("ROLLINGDB_ID=='SAMN02381042'")

,ROLLINGDB_ID,BIOSAMPLE_ACCESSION,SAMPLE,RUN,DB_OF_ORIGIN,ISOLATION_DATE,ISOLATION_COUNTRY,ISOLATION_REGION,SPUTUM/CULTURE,UNPAIRED/PAIRED,Platform,Model,SPECIES,TaxID,ReleaseDate,LoadDate,Num_Runs,Internal_IDs,Combined_Runs
40922,SAMN02381042,SAMN02381042,SRS515098,SRR1049733,PATRIC_20191206,NaN,Uganda,Uganda,1.0,1.0,ILLUMINA,Illumina HiSeq 2000,Mycobacterium tuberculosis UT0106,1408965.0,2013-12-13 18:30:04,2013-12-15 22:58:43,4,NaN,"SRR1049733,SRR1049734,SRR1062932,SRR1140625"
40923,SAMN02381042,SAMN02381042,SRS515098,SRR1049734,PATRIC_20191206,NaN,Uganda,Uganda,1.0,1.0,ILLUMINA,Illumina HiSeq 2000,Mycobacterium tuberculosis UT0106,1408965.0,2013-12-13 18:30:04,2016-11-22 02:04:15,4,NaN,"SRR1049733,SRR1049734,SRR1062932,SRR1140625"
40924,SAMN02381042,SAMN02381042,SRS515098,SRR1062932,PATRIC_20191206,NaN,Uganda,Uganda,1.0,1.0,ILLUMINA,Illumina HiSeq 2000,Mycobacterium tuberculosis UT0106,1408965.0,2013-12-27 21:38:13,2013-12-27 21:40:40,4,NaN,"SRR1049733,SRR1049734,SRR1062932,SRR1140625"
40925,SAMN02381042,SAMN02381042,SRS515098,SRR1140625,PATRIC_20191206,NaN,Uganda,Uganda,1.0,1.0,ILLUMINA,Illumina HiSeq 2000,Mycobacterium tuberculosis UT0106,1408965.0,2014-01-23 04:23:07,2014-01-23 04:24:59,4,NaN,"SRR1049733,SRR1049734,SRR1062932,SRR1140625"


In [ ]:
seqkit pair